In [1]:
import os
import sys
from pathlib import Path

os.environ.pop("DEBUG", None)

env_dir = Path(sys.prefix)

required_paths = [
    env_dir,
    env_dir / "Scripts",
    env_dir / "Library" / "bin",
]

os.environ["PATH"] = os.pathsep.join(
    str(path) for path in required_paths
) + os.pathsep + os.environ.get("PATH", "")

import shutil

print("Meson:", shutil.which("meson"))
print("Ninja:", shutil.which("ninja"))

Meson: D:\YB_Installations\miniconda3\envs\pyme310\Scripts\meson.EXE
Ninja: D:\YB_Installations\miniconda3\envs\pyme310\Scripts\ninja.EXE


In [2]:
import logging

# Hide scyjava conversion-debug messages.
logging.getLogger('scyjava').setLevel(logging.WARNING)
logging.getLogger('scyjava._convert').setLevel(logging.WARNING)

# Hide the headless-mode warning but retain real errors.
logging.getLogger('imagej').setLevel(logging.ERROR)

In [5]:
import os
from oidic import reconstruct
from PYME.IO.DataSources import ArrayDataSource
from PYME.IO.DataSources import BaseDataSource
from PYME.IO.image import ImageStack
from PYME.IO import MetaDataHandler
from PYME.Analysis import piecewiseMapping
from PYME.Analysis import composite
# from PYME.DSView.modules.tiling import Tiler
from PYME.Analysis import deTile
from PYME.Analysis import composite
import matplotlib.pyplot as plt
from tifffile import imwrite
import numpy as np
import time
import PYME.DSView.dsviewer as dsviewer
import imagej_rolling_ball

In [4]:
bg_subtracter = imagej_rolling_ball.BackgroundSubtracter(java_options='-Xmx4g')

Java option: -Xmx4g
ImageJ Version: 2.17.0/1.54p


In [8]:
# Stitch two-color fluorescence, and overlay

path_fluo = 'F:/Acquisition_Yujin_3/Collaboration/Zhangliang/20260728/'

uppercase_alphabet = [chr(i) for i in range(ord('A'), ord('I') + 1)]
uppercase_alphabet2 = [chr(i) for i in range(ord('A'), ord('Z') + 1)]

for i in range(len(uppercase_alphabet)):
    for j in range (0, len(uppercase_alphabet2), 2):
        if i == 0:
            fn_fluo_g = path_fluo + '28_7_series_' + uppercase_alphabet2[j] + '.tiles/raw_frames.h5'
            fn_fluo_r = path_fluo + '28_7_series_' + uppercase_alphabet2[j+1] + '.tiles/raw_frames.h5'
        else:
            fn_fluo_g = path_fluo + '28_7_series_' + uppercase_alphabet[i] + \
                        uppercase_alphabet2[j] + '.tiles/raw_frames.h5'
            fn_fluo_r = path_fluo + '28_7_series_' + uppercase_alphabet[i] + \
                        uppercase_alphabet2[j+1] + '.tiles/raw_frames.h5'
        if not os.path.isfile(fn_fluo_g) or not os.path.isfile(fn_fluo_r):
            continue
        else:
            g = ImageStack(filename = fn_fluo_g)
            r = ImageStack(filename = fn_fluo_r)
        
        x_corr, y_corr = 0.0, 0.0

        tg = piecewiseMapping.frames_to_times(np.arange(g.data.shape[2]), g.events, g.mdh) \
            + g.mdh['Camera.IntegrationTime'] / 2.0
        x0g = g.mdh.getEntry('Positioning.x')
        xmg = piecewiseMapping.GenerateBacklashCorrPMFromEventList(g.events, g.mdh, g.mdh.getEntry('StartTime'), \
                                                                   x0g, b'ScannerXPos', 0, x_corr, in_frames=False)
        xpsg = xmg(tg)
        y0g = g.mdh.getEntry('Positioning.y')
        ymg = piecewiseMapping.GenerateBacklashCorrPMFromEventList(g.events, g.mdh, g.mdh.getEntry('StartTime'), \
                                                                   y0g, b'ScannerYPos', 0, y_corr, in_frames=False)
        ypsg = ymg(tg)
        darkg = g.mdh.getEntry('Camera.ADOffset')
        dtg = deTile.assemble_tiles(g.data, xpsg, ypsg, g.mdh, split=False, skipMoveFrames=False, \
                                    dark=darkg, flat=None, correlate=False)
        mdhg = MetaDataHandler.NestedClassMDHandler(g.mdh)

        tr = piecewiseMapping.frames_to_times(np.arange(r.data.shape[2]), r.events, r.mdh) \
            + r.mdh['Camera.IntegrationTime'] / 2.0
        x0r = r.mdh.getEntry('Positioning.x')
        xmr = piecewiseMapping.GenerateBacklashCorrPMFromEventList(r.events, r.mdh, r.mdh.getEntry('StartTime'), \
                                                                   x0r, b'ScannerXPos', 0, x_corr, in_frames=False)
        xpsr = xmr(tr)
        y0r = r.mdh.getEntry('Positioning.y')
        ymr = piecewiseMapping.GenerateBacklashCorrPMFromEventList(r.events, r.mdh, r.mdh.getEntry('StartTime'), \
                                                                   y0r, b'ScannerYPos', 0, y_corr, in_frames=False)
        ypsr = ymr(tr)
        darkr = r.mdh.getEntry('Camera.ADOffset')
        dtr = deTile.assemble_tiles(r.data, xpsr, ypsr, r.mdh, split=False, skipMoveFrames=False, \
                                    dark=darkr, flat=None, correlate=False)
        mdhr = MetaDataHandler.NestedClassMDHandler(r.mdh)
        ## dtg, dtr: data

        imgg = ImageStack(dtg, mdh=mdhg)
        imgr = ImageStack(dtr, mdh=mdhr)

        images = [(imgr, 0, ''), (imgg, 0, '')]
        overlay = composite.make_composite(images)
        overlay.Save(filename = fn_fluo_r.replace('.tiles/raw_frames.h5', '_fluo.h5'))

filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_A.tiles\raw_frames.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\analysis\28_7_series_A.tiles\raw_frames.h5r
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_B.tiles\raw_frames.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\analysis\28_7_series_B.tiles\raw_frames.h5r


C:\Users\yb262\AppData\Local\Temp\ipykernel_51160\3632051089.py:26: PYMEDeprecationWarning: This will either disappear or change function as we move to a 5D data model. Use the explicit .data_xytc instead, or even better, change to using the 5D model as image.data_xyztc or image.voxels
  tg = piecewiseMapping.frames_to_times(np.arange(g.data.shape[2]), g.events, g.mdh) \
C:\Users\yb262\python-microscopy-python310\PYME\IO\image.py:365: DeprecationWarning: Use voxelsize_nm  property instead
  warnings.warn(DeprecationWarning('Use voxelsize_nm  property instead'))
C:\Users\yb262\python-microscopy-python310\PYME\IO\image.py:1478: DeprecationWarning: The "Save" method is deprecated, please use "save" instead
  warnings.warn(DeprecationWarning('The "Save" method is deprecated, please use "save" instead'))


(3444, 3444, 1) (3444, 3444, 1)
(3444, 3444, 1) (3444, 3444, 1)
Untitled Image 3
(3444, 3444)
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_C.tiles\raw_frames.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\analysis\28_7_series_C.tiles\raw_frames.h5r
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_D.tiles\raw_frames.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\analysis\28_7_series_D.tiles\raw_frames.h5r


C:\Users\yb262\python-microscopy-python310\PYME\IO\image.py:365: DeprecationWarning: Use voxelsize_nm  property instead
  warnings.warn(DeprecationWarning('Use voxelsize_nm  property instead'))
C:\Users\yb262\python-microscopy-python310\PYME\IO\image.py:1478: DeprecationWarning: The "Save" method is deprecated, please use "save" instead
  warnings.warn(DeprecationWarning('The "Save" method is deprecated, please use "save" instead'))


(3444, 3444, 1) (3444, 3444, 1)
(3444, 3444, 1) (3444, 3444, 1)
Untitled Image 3
(3444, 3444)
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_E.tiles\raw_frames.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\analysis\28_7_series_E.tiles\raw_frames.h5r
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_F.tiles\raw_frames.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\analysis\28_7_series_F.tiles\raw_frames.h5r


...

(3444, 3444, 1) (3444, 3444, 1)
(3444, 3444, 1) (3444, 3444, 1)
Untitled Image 32
(3444, 3444)
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_HM.tiles\raw_frames.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\analysis\28_7_series_HM.tiles\raw_frames.h5r
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_HN.tiles\raw_frames.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\analysis\28_7_series_HN.tiles\raw_frames.h5r


C:\Users\yb262\python-microscopy-python310\PYME\IO\image.py:365: DeprecationWarning: Use voxelsize_nm  property instead
  warnings.warn(DeprecationWarning('Use voxelsize_nm  property instead'))
C:\Users\yb262\python-microscopy-python310\PYME\IO\image.py:1478: DeprecationWarning: The "Save" method is deprecated, please use "save" instead
  warnings.warn(DeprecationWarning('The "Save" method is deprecated, please use "save" instead'))


(3444, 3444, 1) (3444, 3444, 1)
(3444, 3444, 1) (3444, 3444, 1)
Untitled Image 32
(3444, 3444)
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_HO.tiles\raw_frames.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\analysis\28_7_series_HO.tiles\raw_frames.h5r
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_HP.tiles\raw_frames.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\analysis\28_7_series_HP.tiles\raw_frames.h5r


C:\Users\yb262\python-microscopy-python310\PYME\IO\image.py:365: DeprecationWarning: Use voxelsize_nm  property instead
  warnings.warn(DeprecationWarning('Use voxelsize_nm  property instead'))
C:\Users\yb262\python-microscopy-python310\PYME\IO\image.py:1478: DeprecationWarning: The "Save" method is deprecated, please use "save" instead
  warnings.warn(DeprecationWarning('The "Save" method is deprecated, please use "save" instead'))


(3444, 3444, 1) (3444, 3444, 1)
(3444, 3444, 1) (3444, 3444, 1)
Untitled Image 3
(3444, 3444)


In [7]:
# OI-DIC reconstruction, rolling-ball background subtraction, and stitching

path = 'F:/Acquisition_Yujin_3/Collaboration/Zhangliang/20260728/oidic/'
fn_oidic_bg = path + 'oidicbg1.h5'
oidicbg = ImageStack(filename = fn_oidic_bg)

uppercase_alphabet = [chr(i) for i in range(ord('A'), ord('H') + 1)]
uppercase_alphabet2 = [chr(i) for i in range(ord('A'), ord('Z') + 1)]

for i in range(len(uppercase_alphabet)):
    for j in range (len(uppercase_alphabet2)):
        if i == 0:
            fn_oidic = path + '28_7_series_' + uppercase_alphabet2[j] + '.h5'
        else:
            fn_oidic = path + '28_7_series_' + uppercase_alphabet[i] + uppercase_alphabet2[j] + '.h5'
        if not os.path.isfile(fn_oidic):
            continue
        else:
            oidic = ImageStack(filename = fn_oidic)
        opd = reconstruct.reconstruct(image_stack=oidic, wavelength=546.0, bias=0.15*546.0, 
                                      shear_distance=70.0, numerical_aperture=1.35, background_stack=oidicbg, 
                                      n_frames=6, reconstruction_type='integrate', shear_bias=0)
#         opd = np.float32(opd[::-1, ::-1, :, :])
        
        # rollong-ball
        opd_rb = np.stack(
            [
                bg_subtracter.rolling_ball_background(
                    img=opd[:, :, 0, t],
                    radius=1000.0,
                    use_paraboloid=True,
                )
                for t in range(opd.shape[3])
            ],
            axis=-1,
        )
        opd_rb = BaseDataSource.XYZTCWrapper(ArrayDataSource.ArrayDataSource(opd_rb), 'XYZTC', \
                                          opd.shape[2], opd.shape[3], 1)
        opd_rb = ImageStack(opd_rb, mdh=oidic.mdh, events=oidic.events)
        opd_stitched = deTile.assemble_tiles_xyztc(opd_rb.data_xyztc, opd_rb.mdh, dark=None)
        
        oidic.mdh['Camera.ROIOriginX'] = 1345
        oidic.mdh['Camera.ROIOriginY'] = 1265
        opd_stitched_ims = ImageStack(opd_stitched[::-1, ::-1], mdh=oidic.mdh)
        
        if j > 0:
            prev_i = i
            prev_j = j - 1
        else:
            prev_i = i - 1
            prev_j = len(uppercase_alphabet2) - 1 # Wrap around to last letter (e.g., Z)
            
        # 3. Generate the previous filename (if we haven't gone past the very first item)
        if prev_i >= 0:
            if prev_i == 0:
                fn_prev = path + '28_7_series_' + uppercase_alphabet2[prev_j] + '.h5'
            else:
                fn_prev = path + '28_7_series_' + uppercase_alphabet[prev_i] + uppercase_alphabet2[prev_j] + '.h5'
        else:
            fn_prev = None
            
        save_path = 'F:/Acquisition_Yujin_3/Collaboration/Zhangliang/20260728/' + fn_prev[len(path):]
        opd_stitched_ims.Save(filename = save_path.replace('.h5', '_opd.h5'))
#         imwrite(fn_oidic.replace('.h5', '_opd.ome.tif'), opd_stitched)

filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\oidic\oidicbg1.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\analysis\oidic\oidicbg1.h5r
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\oidic\28_7_series_C.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\analysis\oidic\28_7_series_C.h5r


C:\Users\yb262\python-microscopy-python310\PYME\IO\image.py:1478: DeprecationWarning: The "Save" method is deprecated, please use "save" instead
  warnings.warn(DeprecationWarning('The "Save" method is deprecated, please use "save" instead'))


(5571, 5571)
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\oidic\28_7_series_E.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\analysis\oidic\28_7_series_E.h5r


C:\Users\yb262\python-microscopy-python310\PYME\IO\image.py:1478: DeprecationWarning: The "Save" method is deprecated, please use "save" instead
  warnings.warn(DeprecationWarning('The "Save" method is deprecated, please use "save" instead'))


(5571, 5571)
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\oidic\28_7_series_G.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\analysis\oidic\28_7_series_G.h5r


...

(5571, 5571)
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\oidic\28_7_series_HO.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\analysis\oidic\28_7_series_HO.h5r


C:\Users\yb262\python-microscopy-python310\PYME\IO\image.py:1478: DeprecationWarning: The "Save" method is deprecated, please use "save" instead
  warnings.warn(DeprecationWarning('The "Save" method is deprecated, please use "save" instead'))


(5571, 5571)
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\oidic\28_7_series_HQ.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\analysis\oidic\28_7_series_HQ.h5r


C:\Users\yb262\python-microscopy-python310\PYME\IO\image.py:1478: DeprecationWarning: The "Save" method is deprecated, please use "save" instead
  warnings.warn(DeprecationWarning('The "Save" method is deprecated, please use "save" instead'))


(5571, 5571)


In [8]:
# Overlay channels

path_fluo = 'F:/Acquisition_Yujin_3/Collaboration/Zhangliang/20260728/'
uppercase_alphabet = [chr(i) for i in range(ord('A'), ord('I') + 1)]
uppercase_alphabet2 = [chr(i) for i in range(ord('A'), ord('Z') + 1)]

for i in range(len(uppercase_alphabet)):
    for j in range (len(uppercase_alphabet2)):
        if i == 0:
            fn_fluo_stitched = path_fluo + '28_7_series_' + uppercase_alphabet2[j] + '_fluo.h5'
            fn_oidic_stitched = path_fluo + '28_7_series_' + uppercase_alphabet2[j] + '_opd.h5'
        else:
            fn_fluo_stitched = path_fluo + '28_7_series_' + uppercase_alphabet[i] + \
                        uppercase_alphabet2[j] + '_fluo.h5'
            fn_oidic_stitched = path_fluo + '28_7_series_' + uppercase_alphabet[i] + \
                        uppercase_alphabet2[j] + '_opd.h5'
        if not os.path.isfile(fn_fluo_stitched) or not os.path.isfile(fn_oidic_stitched):
            continue
        else:
            fluo_stitched = ImageStack(filename = fn_fluo_stitched)
            oidic_stitched = ImageStack(filename = fn_oidic_stitched)
            
        images = [(fluo_stitched, 0, ''), (fluo_stitched, 1, ''), (oidic_stitched, 0, '')]
        overlay = composite.make_composite(images)
        overlay.Save(filename = fn_fluo_stitched.replace('fluo.h5', 'composite.h5'))

filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_B_fluo.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\analysis\20260728\28_7_series_B_fluo.h5r
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_B_opd.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\analysis\20260728\28_7_series_B_opd.h5r
(np.int64(3444), np.int64(3444), 1) (np.int64(3444), np.int64(3444), 1)


C:\Users\yb262\python-microscopy-python310\PYME\IO\image.py:365: DeprecationWarning: Use voxelsize_nm  property instead
  warnings.warn(DeprecationWarning('Use voxelsize_nm  property instead'))


(np.int64(3444), np.int64(3444), 1) (np.int64(3444), np.int64(3444), 1)
(np.int64(3444), np.int64(3444), 1) (np.int64(5571), np.int64(5571), 1)
('Remapping ', 'F:\\Acquisition_Yujin_3\\Collaboration\\Zhangliang\\20260728\\28_7_series_B_opd.h5', False, (np.float64(86752.5), np.float64(81592.5), np.float64(30080.499999999985)), False, True, '', np.float64(64.5), True)


C:\Users\yb262\python-microscopy-python310\PYME\IO\image.py:365: DeprecationWarning: Use voxelsize_nm  property instead
  warnings.warn(DeprecationWarning('Use voxelsize_nm  property instead'))


(np.float64(64.5), np.float64(64.5), np.float64(200.0), (np.int64(5571), np.int64(5571), 1, 1, 1))
28_7_series_B_
(3444, 3444)


...

C:\Users\yb262\python-microscopy-python310\PYME\IO\image.py:365: DeprecationWarning: Use voxelsize_nm  property instead
  warnings.warn(DeprecationWarning('Use voxelsize_nm  property instead'))


(np.int64(3444), np.int64(3444), 1) (np.int64(3444), np.int64(3444), 1)
(np.int64(3444), np.int64(3444), 1) (np.int64(5571), np.int64(5571), 1)
('Remapping ', 'F:\\Acquisition_Yujin_3\\Collaboration\\Zhangliang\\20260728\\28_7_series_HP_opd.h5', False, (np.float64(86752.5), np.float64(81592.5), np.float64(29281.499999999916)), False, True, '', np.float64(64.5), True)


C:\Users\yb262\python-microscopy-python310\PYME\IO\image.py:365: DeprecationWarning: Use voxelsize_nm  property instead
  warnings.warn(DeprecationWarning('Use voxelsize_nm  property instead'))


(np.float64(64.5), np.float64(64.5), np.float64(200.0), (np.int64(5571), np.int64(5571), 1, 1, 1))
28_7_series_HP_
(3444, 3444)


C:\Users\yb262\python-microscopy-python310\PYME\IO\image.py:1478: DeprecationWarning: The "Save" method is deprecated, please use "save" instead
  warnings.warn(DeprecationWarning('The "Save" method is deprecated, please use "save" instead'))


In [9]:
# Concatenate time sequence

path_fluo = 'F:/Acquisition_Yujin_3/Collaboration/Zhangliang/20260728/'
uppercase_alphabet = [chr(i) for i in range(ord('A'), ord('I') + 1)]
uppercase_alphabet2 = [chr(i) for i in range(ord('A'), ord('Z') + 1)]

fn = []
for i in range(len(uppercase_alphabet)):
    for j in range (len(uppercase_alphabet2)):
        if i == 0:
            fn_overlay = path_fluo + '28_7_series_' + uppercase_alphabet2[j] + '_composite.h5'
        else:
            fn_overlay = path_fluo + '28_7_series_' + uppercase_alphabet[i] + \
                        uppercase_alphabet2[j] + '_composite.h5'
        if not os.path.isfile(fn_overlay):
            continue
        else:
            fn.append(fn_overlay)
            
# combine stack, pyme-psf-extraction repo
# see function psf_extraction.plugins.recipes.psf_extraction.CombineBeadStacks
ims = ImageStack(filename=fn[0])
dims = np.asarray(ims.data.shape, dtype=np.long)
dims[2] = 0
dtype_ = ims.data[:,0,0,0].dtype
mdh = ims.mdh
del ims

for fil in fn:
    ims = ImageStack(filename=fil)
    dims[2] += ims.data.shape[2]
    del ims
    
raw_data = np.zeros(shape=tuple(dims), dtype=dtype_)
    
counter = 0
for fil in fn:
    ims = ImageStack(filename=fil)
    z_len = ims.data.shape[2]
    data = ims.data[:,:,:,:]
    data.shape += (1,) * (4 - data.ndim)
    raw_data[:,:,counter:counter+z_len,:] = data
    counter += z_len
            
    del ims
    
new_mdh = None
try:
    new_mdh = MetaDataHandler.NestedClassMDHandler(mdh)
    new_mdh["PSFExtraction.SourceFilenames"] = fn
except Exception as e:
    print(e)
            
combined = ImageStack(data=raw_data, mdh=new_mdh)
combined.Save(filename = fn_overlay.replace('composite.h5', 'composite_concatenate.h5'))

filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_B_composite.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\analysis\20260728\28_7_series_B_composite.h5r


C:\Users\yb262\AppData\Local\Temp\ipykernel_33700\47635959.py:23: PYMEDeprecationWarning: This will either disappear or change function as we move to a 5D data model. Use the explicit .data_xytc instead, or even better, change to using the 5D model as image.data_xyztc or image.voxels
  dims = np.asarray(ims.data.shape, dtype=np.long)


filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_B_composite.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\analysis\20260728\28_7_series_B_composite.h5r
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_D_composite.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\analysis\20260728\28_7_series_D_composite.h5r
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_F_composite.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\analysis\20260728\28_7_series_F_composite.h5r
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_H_composite.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\analysis\20260728\28_7_series_H_composite.h5r
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_J_composite.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\analysis\20260728\28_7_series_J_composite.h5r
filename == F:\Acquisition_Yujin_3\Collaboration\Z

filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_DJ_composite.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\analysis\20260728\28_7_series_DJ_composite.h5r
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_DL_composite.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\analysis\20260728\28_7_series_DL_composite.h5r
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_DN_composite.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\analysis\20260728\28_7_series_DN_composite.h5r
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_DP_composite.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\analysis\20260728\28_7_series_DP_composite.h5r
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_DR_composite.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\analysis\20260728\28_7_series_DR_composite.h5r
filename == F:\Acquisition_Yujin_3\Colla

filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_GT_composite.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\analysis\20260728\28_7_series_GT_composite.h5r
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_GV_composite.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\analysis\20260728\28_7_series_GV_composite.h5r
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_GX_composite.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\analysis\20260728\28_7_series_GX_composite.h5r
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_GZ_composite.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\analysis\20260728\28_7_series_GZ_composite.h5r
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_HB_composite.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\analysis\20260728\28_7_series_HB_composite.h5r
filename == F:\Acquisition_Yujin_3\Colla

filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_CL_composite.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\analysis\20260728\28_7_series_CL_composite.h5r
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_CN_composite.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\analysis\20260728\28_7_series_CN_composite.h5r
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_CP_composite.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\analysis\20260728\28_7_series_CP_composite.h5r
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_CR_composite.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\analysis\20260728\28_7_series_CR_composite.h5r
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_CT_composite.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\analysis\20260728\28_7_series_CT_composite.h5r
filename == F:\Acquisition_Yujin_3\Colla

filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_FT_composite.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\analysis\20260728\28_7_series_FT_composite.h5r
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_FV_composite.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\analysis\20260728\28_7_series_FV_composite.h5r
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_FX_composite.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\analysis\20260728\28_7_series_FX_composite.h5r
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_FZ_composite.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\analysis\20260728\28_7_series_FZ_composite.h5r
filename == F:\Acquisition_Yujin_3\Collaboration\Zhangliang\20260728\28_7_series_GB_composite.h5
F:\Acquisition_Yujin_3\Collaboration\Zhangliang\analysis\20260728\28_7_series_GB_composite.h5r
filename == F:\Acquisition_Yujin_3\Colla

C:\Users\yb262\python-microscopy-python310\PYME\IO\image.py:1478: DeprecationWarning: The "Save" method is deprecated, please use "save" instead
  warnings.warn(DeprecationWarning('The "Save" method is deprecated, please use "save" instead'))


(3444, 3444)
